<a href="https://colab.research.google.com/github/hossain8078/Machine-Learning-and-Deep-Learning/blob/main/Lab_7_Cloud_Deployment_Research_AWS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab Exercise 7: Researching Cloud Deployment Options

**Objective:** In this lab, you will research and compare different cloud services for deploying machine learning models. This is a research-based exercise designed to familiarize you with the cloud deployment landscape.

### Instructions

Choose one of the following cloud providers: **Amazon Web Services (AWS)**, **Google Cloud Platform (GCP)**, or **Microsoft Azure**.

For your chosen provider, research and write a short report (1-2 pages) that answers the following questions:

1.  **PaaS (Platform-as-a-Service) Options:**
    - Identify and describe a PaaS offering for deploying a web application (like a Flask API). Examples include AWS Elastic Beanstalk, Google App Engine, or Azure App Service.
    - What are the key features of this service?
    - What are the pros and cons of using this service for deploying a machine learning model?
2.  **CaaS (Containers-as-a-Service) Options:**
    - Identify and describe a CaaS offering for deploying Docker containers. Examples include AWS ECS, Google Cloud Run, or Azure Container Instances.
    - How does this service work?
    - What are the pros and cons of using this service compared to the PaaS option?
3.  **FaaS (Functions-as-a-Service) / Serverless Options:**
    - Identify and describe a serverless offering. Examples include AWS Lambda, Google Cloud Functions, or Azure Functions.
    - How would you deploy a machine learning model using this service?
    - What are the benefits and drawbacks of the serverless approach?
4.  **Recommendation:**
    - Based on your research, which service would you recommend for deploying a simple machine learning model for a small-scale application? Justify your choice.

---
# Report: Deploying ML Models on Amazon Web Services (AWS)

**Chosen provider:** Amazon Web Services

**Scenario used throughout:** the Iris `DecisionTreeClassifier` from the example notebook, saved as `iris_model.pkl` and served by the Flask `app.py` with a `/predict` endpoint.

| Model | AWS service | You manage | AWS manages |
|---|---|---|---|
| PaaS | Elastic Beanstalk | Code + small config | EC2 servers, OS patches, load balancer, auto scaling |
| CaaS | Amazon ECS on AWS Fargate | Container image + task definition | Servers, container placement, scaling |
| FaaS | AWS Lambda | One handler function | Everything else |

Note (Overview): AWS gives three ways to host the same model. Going down the table, you manage less and AWS manages more. The trade-off is control: Beanstalk still runs EC2 servers you can see and tune, ECS lets you define the whole environment through a container, and Lambda hides the infrastructure completely.

## 1. PaaS: AWS Elastic Beanstalk

**What it is.** Elastic Beanstalk is AWS's managed platform for web applications. You upload your code (a zip file or `eb deploy`), and Beanstalk creates and manages the resources underneath: EC2 instances, an Auto Scaling group, an optional Elastic Load Balancer, security groups and monitoring. The service itself is free; you pay only for the EC2 and other resources it creates.

**Key features**
- Ready-made platforms for Python, Java, Node.js, .NET, Go, PHP, Ruby and Docker.
- Auto scaling and load balancing based on CPU, traffic or other CloudWatch metrics.
- Deployment policies (all-at-once, rolling, immutable, blue/green) for safe model updates and easy rollback.
- Health monitoring, logs and alarms through Amazon CloudWatch.
- Full access to the underlying EC2 instances (instance type, SSH, `.ebextensions` config) when needed.

**Deploying the Flask model** needs `app.py`, `iris_model.pkl`, `requirements.txt` and a `Procfile`:
```text
# Procfile
web: gunicorn --bind :8000 --workers 2 app:app
```
```bash
eb init -p python-3.12 iris-api --region us-east-1
eb create iris-env --single --instance-types t3.micro   # single instance, no load balancer
eb deploy                                               # push later model updates
```

**Pros for ML models**
- Fastest path from the notebook's Flask app to a public URL; the code runs unchanged and no Docker knowledge is needed.
- Scaling, health checks, logs and rolling updates come out of the box.
- You can choose any EC2 instance type, including larger memory or GPU instances, which App-style PaaS products often don't allow.

**Cons for ML models**
- Always runs at least one EC2 instance, so it costs money even with zero traffic (no scale to zero).
- Scaling is slower (new EC2 instances take minutes to start) than containers or functions.
- Less control over the OS and stack than raw EC2 or containers, and configuration is AWS-specific (vendor lock-in).

Note: Elastic Beanstalk is the easiest way to put the Flask model online on AWS. You upload the code and AWS builds the servers around it. It suits a small model like the Iris tree, but you pay for at least one server all the time, and scaling up takes minutes.

## 2. CaaS: Amazon ECS on AWS Fargate

**What it is.** Amazon Elastic Container Service (ECS) is AWS's container orchestration service. With the **Fargate** launch type, AWS runs the containers for you, so there are no EC2 servers to manage. (AWS also offers EKS for teams that want Kubernetes.)

**How it works**
1. Write a `Dockerfile` that packages Python, the dependencies, `app.py` and `iris_model.pkl`.
2. Build the image and push it to **Amazon ECR** (Elastic Container Registry).
3. Create a **task definition**: which image to run, how much CPU and memory, which port, and the IAM role.
4. Create an ECS **service** that keeps the chosen number of tasks running, replaces failed ones, and registers them with an **Application Load Balancer**, which gives the public endpoint.
5. Add **Service Auto Scaling** rules (for example, keep average CPU at 60%) to add or remove tasks. Fargate bills per second for the vCPU and memory each task uses.

```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py iris_model.pkl ./
EXPOSE 8000
CMD ["gunicorn", "--bind", "0.0.0.0:8000", "--workers", "2", "app:app"]
```
```bash
aws ecr create-repository --repository-name iris-api
docker build -t iris-api .
docker tag iris-api:latest <account-id>.dkr.ecr.us-east-1.amazonaws.com/iris-api:latest
docker push <account-id>.dkr.ecr.us-east-1.amazonaws.com/iris-api:latest
# then create the task definition and service (console, AWS CLI or AWS Copilot)
```

**Pros compared with Elastic Beanstalk (PaaS)**
- **Any library or OS package**: the container defines the environment, so heavy ML stacks (PyTorch, TensorFlow, system libraries) are easy.
- **Portability**: the same image runs on a laptop, on EKS, or on another cloud, which reduces lock-in.
- **Faster, finer scaling**: containers start in seconds and resources are set per task.
- **Many models side by side**: each model can be its own service, sharing one cluster and load balancer.

**Cons compared with Elastic Beanstalk (PaaS)**
- More to learn and set up: Docker, ECR, task definitions, services, load balancer, networking (VPC, subnets, security groups).
- You maintain the image (base image updates, security patches).
- No scale to zero: at least one task plus the load balancer is billed all the time, which is heavy for a tiny app.

Note: ECS on Fargate runs the same Flask app inside a Docker container. Compared with Beanstalk, it gives more freedom (any library, portable image, faster scaling, many models side by side). The cost is a lot more setup work and an always-on bill, so it pays off for bigger or growing ML systems, not a single small model.

## 3. FaaS / Serverless: AWS Lambda

**What it is.** Lambda runs your code in response to events without any servers to manage. You write a handler function, and AWS runs it on demand and bills per request and per millisecond of run time. Triggers include HTTP calls (through a **Lambda Function URL** or **Amazon API Gateway**), S3 file uploads, SQS queues and EventBridge schedules.

**How to deploy the ML model**
1. Write a handler that loads the model **once, outside the handler**, so warm invocations reuse it:
```python
# lambda_function.py
import json, joblib

model = joblib.load("iris_model.pkl")          # loaded once per container (cold start)
NAMES = ["setosa", "versicolor", "virginica"]

def lambda_handler(event, context):
    features = json.loads(event["body"])["features"]
    pred = int(model.predict([features])[0])
    return {"statusCode": 200, "body": json.dumps({"prediction": NAMES[pred]})}
```
2. Package it. scikit-learn, NumPy and SciPy together come close to Lambda's 250 MB limit for zip packages, so the easiest route is a **container image** (up to 10 GB) built from AWS's base image:
```dockerfile
FROM public.ecr.aws/lambda/python:3.12
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY lambda_function.py iris_model.pkl ./
CMD ["lambda_function.lambda_handler"]
```
3. Push the image to ECR, create the function from it (for example, 1024 MB memory and a 30 s timeout), and add a **Function URL** or an **API Gateway** route so clients can POST to it.

The same function could instead be triggered by an S3 upload to run **batch predictions** on new data files.

**Benefits**
- No servers at all, and it scales automatically from zero to thousands of parallel requests.
- Pay only per request and run time; the monthly free tier (1 million requests) easily covers a student or small app.
- Event triggers make it easy to automate ML workflows (score new files in S3, retrain on a schedule).

**Drawbacks**
- **Cold starts**: loading Python, scikit-learn and the model adds noticeable latency to the first request after idle time.
- Hard limits: 15-minute maximum run time, up to 10 GB memory, package size limits and no GPUs, so it's unsuitable for large deep-learning models.
- Stateless and harder to debug and monitor; the Flask app must be rewritten as a handler (or wrapped with an adapter).

Note: Lambda is the cheapest option for occasional or event-driven predictions, because you pay only per call and nothing when idle. Cold starts, size and time limits, no GPUs and the need to rewrite the Flask app make it weaker for a busy, low-latency API or a large model.

## 4. Recommendation

**For a simple ML model in a small-scale application, I recommend AWS Elastic Beanstalk** (single-instance environment).

**Justification**
1. **Simplest to deploy**: the Flask `app.py` and `iris_model.pkl` from the example notebook run unchanged. With a `Procfile` and `eb create`, the model is online. ECS needs Docker, ECR, a task definition, a load balancer and networking; Lambda needs the code rewritten as a handler plus a container image for scikit-learn.
2. **Low, predictable cost**: one small instance (for example `t3.micro`, without a load balancer) costs a few dollars a month. ECS on Fargate would add an always-on load balancer on top of the task cost.
3. **Enough capacity**: a decision tree predicts in milliseconds, so one instance easily handles small-scale traffic, and auto scaling can be switched on later.
4. **Managed operations**: AWS handles OS patching, health checks, logs (CloudWatch) and safe rolling or blue/green updates when a new model version is released.
5. **Room to grow**: you can pick a bigger instance type or add a load balancer and auto scaling without changing the code.

**When I would choose differently**
- **AWS Lambda** if traffic is very low or irregular (cost close to zero thanks to scale to zero) or predictions are event-driven, such as scoring each new file uploaded to S3.
- **ECS on Fargate** if the application grows to several models, needs custom system libraries or GPUs, or the team already uses Docker.

**Production checklist (from the lecture):** protect the endpoint (IAM, API keys or API Gateway authorizers), enable HTTPS, monitor latency and errors with CloudWatch, watch for data drift, and keep models versioned (for example, in S3 or the SageMaker Model Registry) so a bad release can be rolled back.

*Note: service limits, prices and free-tier amounts change over time; check the current AWS documentation before deploying.*

Overall result: Elastic Beanstalk** gives the best balance for a small ML app on AWS. It needs the least new work (the Flask code runs as-is), costs a small fixed amount each month, and AWS handles the servers. Lambda is the better choice only for very sporadic or event-driven use, and ECS only when the system grows larger.